In [6]:
from pymongo import MongoClient
from datetime import timezone
from hansard_common import ROOT, fetch, build_proxies, discover_hansard_day_urls, utc_now
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import logging
from pathlib import Path
from dotenv import load_dotenv

In [19]:
LOG_PATH = Path("LOGS/incremental_scraper.log")
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=LOG_PATH,
    filemode="a",
    format=">>> %(levelname)s | %(asctime)s | %(message)s",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

load_dotenv(override=True)

False

In [21]:
client = MongoClient("mongodb://localhost:27017/", tz_aware=True, tzinfo=timezone.utc)
db = client["case-scraping"]       
html_col = db["hansard-html-snapshots"]         

In [9]:
# create unique index on url to prevent duplicates across the collection to later compare the urls against the ones already scraped 
html_col.create_index([("url", 1)], unique=True, name="url_unique")

'url_unique'

In [10]:
# find the recent house documents page url from the root page
def discover_recent_house_documents(root_html: bytes | str, root_url: str = ROOT) -> str:
    soup = BeautifulSoup(root_html, "html.parser")

    for a in soup.find_all("a", href=True):
        text = a.get_text(" ", strip=True).lower()
        if "recent house documents" in text:
            return urljoin(root_url, a["href"])

    logger.error("Could not find Recent House documents link on root page")
    raise RuntimeError("Could not find Recent House documents link on root page")

In [11]:
# function to find the urls already scraped
def existing_day_urls(urls: list[str]) -> set[str]:
    docs = html_col.find(
        {"url": {"$in": urls}},
        {"_id": 0, "url": 1}
    )
    return {doc["url"] for doc in docs}

In [ ]:
def main():
    proxies = build_proxies()
    root_resp = fetch(ROOT, proxies=proxies)
    if not root_resp or not root_resp.ok or not root_resp.content:
        logger.error("Failed to fetch root page: %s", ROOT)
        raise RuntimeError(f"Failed to fetch root page: {ROOT}")
    logger.info("Fetched root page successfully")

    recent_session_url = discover_recent_house_documents(root_resp.content)
    if not recent_session_url:
        logger.error("Stopping run because recent house documents URL could not be discovered")
        raise RuntimeError("Stopping run because recent house documents URL could not be discovered")
    else:
        session_resp = fetch(recent_session_url, proxies=proxies)
        if not session_resp or not session_resp.ok or not session_resp.content:
            logger.error("Failed to fetch recent session page: %s", recent_session_url)
            raise RuntimeError(f"Failed to fetch recent session page: {recent_session_url}")
        else:
            day_urls = sorted(set(discover_hansard_day_urls(session_resp.content)))
            logger.info("Discovered %d unique day URLs on recent session page", len(day_urls))
            
    known_urls = existing_day_urls(day_urls) #urls a;ready present in the db collection
    logger.info("Found %d already-known URLs in Mongo", len(known_urls))
    new_day_urls = [u for u in day_urls if u not in known_urls]
    logger.info("Found %d new day URLs to scrape", len(new_day_urls))

    for day_url in new_day_urls:
        resp = fetch(day_url, proxies=proxies)
        if not resp or not resp.ok or not resp.content:
            logger.warning("Failed day: %s", day_url)
            continue

        html_col.update_one(
            {"url": day_url},
            {
                "$setOnInsert": {
                    "url": day_url,
                    "batch": "incremental_run",
                    "content": resp.content,
                    "fetched_at": utc_now(),
                    "final_url": resp.final_url,
                    "headers": resp.headers,
                    "status": resp.status,
                    "parsed": False,
                    "parsed_at": None,
                    "parse_error": None,

                }
            },
            upsert=True
        )
        logger.info("Stored new day page: %s", day_url)

In [25]:
if __name__ == "__main__":
    try:
        main()
    except Exception:
        logger.exception("Unhandled error in incremental Hansard scraper")
        raise